# Data Access
#

In [0]:
spark.conf.set(
    "fs.azure.account.key.nyctaxisa389.dfs.core.windows.net",
    "uLTO5foTmQB6SjN776XvoV5tebC72zSRWxeRNo8+NliC3hP0eP11ij2tYVfo0hK7UxjFB2oOCIDc+AStDK2cHA=="
)

# Database creation

In [0]:
%sql
CREATE DATABASE gold

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7060853023064899>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE DATABASE gold\n')

File /databricks/python/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2478, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2476 with self.builtin_trap:
   2477     args = (magic_arg_s, cell)
-> 2478     result = fn(*args, **kwargs)
   2480 # The code below prevents the output from being displayed
   2481 # when using magics with decodator @output_can_be_silenced
   2482 # when the last Python token in the expression is a ';'.
   2483 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:124, in SqlMagic.sql(self, line, cell)
    118     self.logger.logDriverEvent({
    119         "eventType": "spark-connect-sql-end

# Data Reading and Writing and Creating Delta Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Storage Variable**

In [0]:
silver = 'abfss://silver@nyctaxisa389.dfs.core.windows.net'
gold =   'abfss://gold@nyctaxisa389.dfs.core.windows.net'

**Data Zone**

In [0]:
df_zone = spark.read.format('parquet')\
          .option('inferSchema', True)\
          .option('Header', True)\
          .load(f'{silver}/trip_zone')

In [0]:
df_zone.display()

LocationID,Borough,Zone,service_zone,zone1,zone2
1,EWR,Newark Airport,EWR,Newark Airport,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Astoria,Boro Zone,Astoria,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null
9,Queens,Auburndale,Boro Zone,Auburndale,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null


In [0]:
df_zone.write.format('delta')\
    .mode('append')\
    .option('path', f'{gold}/trip_zone')\
    .saveAsTable('gold.trip_zone')

In [0]:
%sql
select * from gold.trip_zone

LocationID,Borough,Zone,service_zone,zone1,zone2
1,EWR,Newark Airport,EWR,Newark Airport,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Astoria,Boro Zone,Astoria,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null
9,Queens,Auburndale,Boro Zone,Auburndale,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null


**Trip Type**

In [0]:
df_type = spark.read.format('parquet')\
          .option('inferSchema', True)\
          .option('Header', True)\
          .load(f'{silver}/trip_type')

In [0]:
df_type.display()

trip_type,description
1,Street-hail
2,Dispatch


In [0]:
df_type.write.format('Delta')\
    .mode('append')\
    .option('path', f'{gold}/trip_type')\
    .saveAsTable('gold.trip_type')

In [0]:
%sql
select * from gold.trip_type

trip_type,description
1,Street-hail
2,Dispatch
1,Street-hail
2,Dispatch


**Trip Data**

In [0]:
df_trip = spark.read.format('parquet')\
          .option('inferSchema', True)\
          .option('Header', True)\
          .load(f'{silver}/trip2024data')

In [0]:
df_trip.display()

VendorID,PULocationID,fare_amount,total_amount
2,65,9.3,13.8
2,7,7.2,11.64
2,74,6.5,9.0
2,75,25.4,32.9
2,256,12.1,17.52
1,210,9.3,12.8
2,66,19.8,28.05
2,95,13.5,16.0
2,24,12.8,21.05
2,210,8.0,9.0


In [0]:
df_trip.write.format('Delta')\
    .mode('append')\
    .option('path', f'{gold}/tripsdata')\
    .saveAsTable('gold.tripsdata')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8749461061111209>, line 4
      1 df_trip.write.format('Delta')\
      2     .mode('append')\
      3     .option('path', f'{gold}/tripsdata')\
----> 4     .saveAsTable('gold.tripsdata')

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:702, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    700 self._write.table_name = name
    701 self._write.table_save_method = "save_as_table"
--> 702 self._spark.client.execute_command(
    703     self._write.command(self._spark.client), self._write.observations
    704 )

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1208, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1206     req.user_context.user_id = self._user_id
   1207 req.plan.command.CopyFrom(command)
-> 1208 dat

In [0]:
%sql
select * from gold.tripsdata

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8749461061111210>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from gold.tripsdata\n')

File /databricks/python/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2478, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2476 with self.builtin_trap:
   2477     args = (magic_arg_s, cell)
-> 2478     result = fn(*args, **kwargs)
   2480 # The code below prevents the output from being displayed
   2481 # when using magics with decodator @output_can_be_silenced
   2482 # when the last Python token in the expression is a ';'.
   2483 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:124, in SqlMagic.sql(self, line, cell)
    118     self.logger.logDriverEvent({
    119         "eventType": "spark-connect